# M6 round 2 — three-model visual-saturation + encoder-level mechanism

Reproduces the headline tables from `docs/insights/m6_r2_cross_model.md`:
- 3-model paired delta vs label-free (Qwen / LLaVA / InternVL3)
- Visual-saturation correlate: PMR(_nolabel) by object_level
- H7 GAR-by-label cross-model (3-of-3)
- LLaVA encoder/LM/behavior boomerang gap (M3/M4 cross-model)
- LLaVA FC "A" bias is at the logit level (r2c)

In [1]:
import sys
import json
import math
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

OUT = PROJECT_ROOT / 'outputs'
Q_LBL = pd.read_parquet(OUT / 'mvp_full_20260424-094103_8ae1fa3d' / 'predictions_scored.parquet')
Q_LBL = Q_LBL[Q_LBL.prompt_variant == 'open']
Q_NL  = pd.read_parquet(OUT / 'label_free_20260425-031430_315c5318' / 'predictions_scored.parquet')
L_LBL = pd.read_parquet(OUT / 'cross_model_llava_20260425-035506_7ff0256b' / 'predictions_scored.parquet')
L_NL  = pd.read_parquet(OUT / 'cross_model_llava_label_free_20260425-040821_39e68cd4' / 'predictions_scored.parquet')
I_LBL = pd.read_parquet(OUT / 'cross_model_internvl3_20260425-051009_fc710e85' / 'predictions_scored.parquet')
I_NL  = pd.read_parquet(OUT / 'cross_model_internvl3_label_free_20260425-053116_ea0a07c5' / 'predictions_scored.parquet')
print(f'Qwen      open n={len(Q_LBL)}, _nolabel n={len(Q_NL)}')
print(f'LLaVA-1.5 open n={len(L_LBL)}, _nolabel n={len(L_NL)}')
print(f'InternVL3 open n={len(I_LBL)}, _nolabel n={len(I_NL)}')

Qwen      open n=1440, _nolabel n=480
LLaVA-1.5 open n=1440, _nolabel n=480
InternVL3 open n=1440, _nolabel n=480


## Headline 1 — three-model paired delta

The visual-saturation hypothesis predicts: as `PMR(_nolabel)` rises,
headroom for label contribution falls, so paired deltas shrink toward 0
(or sign-mix). Three points span the prediction:

In [2]:
def paired_delta(labeled, label_free):
    nl = label_free.groupby('sample_id').pmr.mean().rename('_nolabel')
    rows = {}
    for lab in ['ball', 'circle', 'planet']:
        lbl = labeled[labeled.label == lab].groupby('sample_id').pmr.mean().rename(lab)
        j = pd.concat([nl, lbl], axis=1).dropna()
        rows[lab] = float((j[lab] - j._nolabel).mean())
    return rows

deltas = pd.DataFrame({
    'Qwen2.5-VL': paired_delta(Q_LBL, Q_NL),
    'LLaVA-1.5':  paired_delta(L_LBL, L_NL),
    'InternVL3':  paired_delta(I_LBL, I_NL),
}).T.round(3)
deltas

,ball,circle,planet
Qwen2.5-VL,0.006,-0.065,0.006
LLaVA-1.5,0.475,0.173,0.244
InternVL3,0.010,0.010,0.010


## Headline 2 — `PMR(_nolabel)` by object_level (visual-only physics)

InternVL3 sits at the upper bound (≥0.97 everywhere), Qwen near it,
LLaVA far below.

In [3]:
pd.DataFrame({
    'Qwen2.5-VL': Q_NL.groupby('object_level').pmr.mean(),
    'LLaVA-1.5':  L_NL.groupby('object_level').pmr.mean(),
    'InternVL3':  I_NL.groupby('object_level').pmr.mean(),
}).round(3)

,Qwen2.5-VL,LLaVA-1.5,InternVL3
object_level,,,
filled,0.933,0.317,0.967
line,0.942,0.142,0.992
shaded,0.942,0.592,1.000
textured,0.975,0.483,1.000


## Headline 3 — H7 GAR by label (3-of-3 cross-model)

In [4]:
pd.DataFrame({
    'Qwen2.5-VL': Q_LBL.groupby('label').gar.mean(),
    'LLaVA-1.5':  L_LBL.groupby('label').gar.mean(),
    'InternVL3':  I_LBL.groupby('label').gar.mean(),
}).round(3)

,Qwen2.5-VL,LLaVA-1.5,InternVL3
label,,,
ball,0.706,0.356,0.816
circle,0.753,0.153,0.791
planet,0.319,0.072,0.431


## r2b — LLaVA cross-model M3/M4: encoder vs LM AUC

The boomerang gap that exists in Qwen does not exist in LLaVA, because
the LLaVA vision encoder is the bottleneck.

In [5]:
QWEN_RUN  = OUT / 'mvp_full_20260424-094103_8ae1fa3d'
LLAVA_CAP = OUT / 'cross_model_llava_capture_20260425-054821_65214a5d'

qwen_vis  = pd.read_csv(QWEN_RUN  / 'probing_vision' / 'layer_sweep_open.csv')
llava_vis = pd.read_csv(LLAVA_CAP / 'probing_vision' / 'layer_sweep_open.csv')
qwen_lm   = pd.read_csv(QWEN_RUN  / 'probing_lm'     / 'layer_sweep_open.csv')
llava_lm  = pd.read_csv(LLAVA_CAP / 'probing_lm'     / 'layer_sweep_open.csv')

vis = pd.DataFrame({
    'layer': sorted(set(qwen_vis.layer.tolist() + llava_vis.layer.tolist())),
}).set_index('layer')
vis['Qwen vision AUC']  = qwen_vis.set_index('layer').auc_mean
vis['LLaVA vision AUC'] = llava_vis.set_index('layer').auc_mean
vis.round(3)

,Qwen vision AUC,LLaVA vision AUC
layer,,
3,0.980,0.707
7,0.985,0.731
11,0.986,0.726
15,0.979,0.732
19,0.986,0.715
23,0.986,0.728
27,0.986,NaN
31,0.985,NaN


In [6]:
lm = pd.DataFrame({
    'layer': sorted(set(qwen_lm.layer.tolist() + llava_lm.layer.tolist())),
}).set_index('layer')
lm['Qwen LM AUC']  = qwen_lm.set_index('layer').auc_mean
lm['LLaVA LM AUC'] = llava_lm.set_index('layer').auc_mean
lm.round(3)

,Qwen LM AUC,LLaVA LM AUC
layer,,
5,0.986,0.732
10,0.986,0.753
15,0.986,0.747
20,0.985,0.748
25,0.986,0.736


In [7]:
# Boomerang summary table
summary = pd.DataFrame({
    'Qwen': [qwen_vis.auc_mean.mean(), qwen_lm.auc_mean.mean(), Q_LBL.pmr.mean()],
    'LLaVA': [llava_vis.auc_mean.mean(), llava_lm.auc_mean.mean(), L_LBL.pmr.mean()],
}, index=['vision encoder AUC (mean over layers)', 'LM AUC (mean over layers)', 'behavioral PMR (open)'])
summary.round(3)

,Qwen,LLaVA
vision encoder AUC (mean over layers),0.984,0.723
LM AUC (mean over layers),0.986,0.743
behavioral PMR (open),0.931,0.681


## r2c — LLaVA FC "A" bias is at the logit level

Re-derive `logit_argmax` over the four FC option letters from the
stored `option_logits` (post temperature/top_p warping). For LLaVA,
even the underlying logit distribution is locked to A.

In [8]:
import collections

def fc_logit_argmax(jsonl_path):
    counts = collections.Counter()
    n_real_dist = collections.Counter()
    with open(jsonl_path) as f:
        for line in f:
            r = json.loads(line)
            if not r.get('prompt_variant', '').startswith('forced_choice'):
                continue
            ol = r.get('option_logits') or {}
            real = {k: v for k, v in ol.items() if v != float('-inf') and not math.isinf(v)}
            n_real_dist[len(real)] += 1
            if real:
                counts[max(real.items(), key=lambda x: x[1])[0]] += 1
    return counts, n_real_dist

rows = []
for label, path in [
    ('Qwen M2 FC labeled',  OUT / 'mvp_full_20260424-094103_8ae1fa3d' / 'predictions.jsonl'),
    ('Qwen FC label-free',  OUT / 'fc_label_free_qwen_20260425-042817_eec92f1a' / 'predictions.jsonl'),
    ('LLaVA FC label-free', OUT / 'fc_label_free_llava_20260425-044517_81ae56d5' / 'predictions.jsonl'),
]:
    counts, n_real = fc_logit_argmax(path)
    total = sum(counts.values())
    rows.append({
        'run': label,
        'n': total,
        'argmax=A %': counts['A'] / total * 100 if total else 0,
        'argmax=B %': counts.get('B', 0) / total * 100 if total else 0,
        'argmax=D %': counts.get('D', 0) / total * 100 if total else 0,
        'rows with only 1 letter post top_p': n_real.get(1, 0),
    })
pd.DataFrame(rows).set_index('run').round(2)

,n,argmax=A %,argmax=B %,argmax=D %,rows with only 1 letter post top_p
run,,,,,
Qwen M2 FC labeled,1440,65.14,0.35,34.51,736
Qwen FC label-free,480,77.29,15.62,7.08,278
LLaVA FC label-free,480,100.00,0.00,0.00,430


LLaVA's logit_argmax is 100% A. 90 % of LLaVA rows have only A surviving
the top_p=0.95 filter at the first generated step — the underlying
probability of A is ≥0.95 in 90 % of cases. The bias is at the logit
level, not introduced by greedy sampling.